In [0]:
%sql
-- Setup

CREATE CATALOG IF NOT EXISTS jf MANAGED LOCATION 'abfss://jfcontainer@sauksdatabricksdata.dfs.core.windows.net/cdc';

CREATE SCHEMA IF NOT EXISTS jf.cdc;

In [0]:
# Sample Data

catalog = "jf"
schema = "cdc"
employees_cdf_table = "employees_cdf"

def write_employees_cdf_to_delta():
    data = [
        (1, "Alex", "chef", "FR", "INSERT", 1),
        (2, "Jessica", "owner", "US", "INSERT", 2),
        (3, "Mikhail", "security", "UK", "INSERT", 3),
        (4, "Gary", "cleaner", "UK", "INSERT", 4),
        (5, "Chris", "owner", "NL", "INSERT", 6),
        # out of order update, this should be dropped from SCD Type 1
        (5, "Chris", "manager", "NL", "UPDATE", 5),
        (6, "Pat", "mechanic", "NL", "DELETE", 8),
        (6, "Pat", "mechanic", "NL", "INSERT", 7)
    ]
    columns = ["id", "name", "role", "country", "operation", "sequenceNum"]
    df = spark.createDataFrame(data, columns)
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{employees_cdf_table}")

write_employees_cdf_to_delta()

In [0]:
%sql
SELECT *
FROM jf.cdc.employees_cdf

In [0]:
# Run DLT Pipeline

In [0]:
%sql
SELECT *
FROM jf.cdc.employees_current

In [0]:
%sql
SELECT *
FROM jf.cdc.employees_historical